# Pushdown query latency

Times one rewritten query per 0.4.1 pushdown filter. Fill in the configuration for the CDF project and the Unity Catalog view you want to measure.

Each query cell prints the SQL result and the seconds spent in `collect()`. The call is `generator.rewrite_query`, so the filter is sent to CDF. A query on the registered view reads every row and filters in Spark.

The view must already be registered. This notebook does not seed data and does not replace the function or the view.

Leave a filter setting empty to skip that query. A pass means the rewriter produced a UDTF call and the call returned. Counts come back in `count_externalId`. `min` / `max` come back in `min_<column>` / `max_<column>`.


## 1. Install 0.4.1


In [ ]:
%pip install --upgrade "cognite-pygen-spark==0.4.1" "cognite-databricks==0.4.1"


In [ ]:
%restart_python


## 2. Configuration

`TOML_PATH` is the settings file. Client credentials stay in the file named by `credentials`.
That credentials file needs `[cognite]` with `project`, `cdf_cluster`, `tenant_id` (or `idp_tenant_id`), `client_id`, and `client_secret`.

`schema` and `secret_scope` can be omitted. They are derived from the data model.


In [ ]:
import tomllib
from pathlib import Path

TOML_PATH = ""  # settings TOML. credentials = path to the CDF client TOML

if not TOML_PATH:
    raise ValueError("Set TOML_PATH to the settings TOML")

with Path(TOML_PATH).open("rb") as handle:
    settings = tomllib.load(handle)

CREDENTIALS_PATH = str(settings.get("credentials") or TOML_PATH)
OUTPUT_DIR = str(settings.get("output_dir") or "/local_disk0/tmp/pygen_udtf")

catalog_settings = settings.get("unity_catalog") or {}
model_settings = settings.get("data_model") or {}
filter_settings = settings.get("filters") or {}

CATALOG = str(catalog_settings.get("catalog") or "")
SCHEMA = str(catalog_settings.get("schema") or "")
SECRET_SCOPE = str(catalog_settings.get("secret_scope") or "")
DATA_MODEL_SPACE = str(model_settings.get("space") or "")
DATA_MODEL_EXTERNAL_ID = str(model_settings.get("external_id") or "")
DATA_MODEL_VERSION = str(model_settings.get("version") or "")
VIEW_EXTERNAL_ID = str(model_settings.get("view") or "")

INSTANCE_SPACES = [str(value) for value in filter_settings.get("instance_spaces") or []]
EXTERNAL_IDS = [str(value) for value in filter_settings.get("external_ids") or []]
STRING_COLUMN = str(filter_settings.get("string_column") or "")
STRING_VALUES = [str(value) for value in filter_settings.get("string_values") or []]
NULLABLE_COLUMN = str(filter_settings.get("nullable_column") or "")
NUMERIC_COLUMN = str(filter_settings.get("numeric_column") or "")
NUMERIC_LOW = filter_settings.get("numeric_low")
NUMERIC_HIGH = filter_settings.get("numeric_high")

required = {
    "credentials": CREDENTIALS_PATH,
    "catalog": CATALOG,
    "data_model.space": DATA_MODEL_SPACE,
    "data_model.external_id": DATA_MODEL_EXTERNAL_ID,
    "data_model.version": DATA_MODEL_VERSION,
    "data_model.view": VIEW_EXTERNAL_ID,
}
missing = [name for name, value in required.items() if not value]
if missing:
    raise ValueError("Settings TOML is missing " + ", ".join(missing))

if not SCHEMA:
    SCHEMA = f"{DATA_MODEL_SPACE}_{DATA_MODEL_EXTERNAL_ID.lower()}_{DATA_MODEL_VERSION}"
if not SECRET_SCOPE:
    SECRET_SCOPE = f"cdf_{DATA_MODEL_SPACE}_{DATA_MODEL_EXTERNAL_ID.lower()}"
VIEW = f"{CATALOG}.{SCHEMA}.{VIEW_EXTERNAL_ID}"
print("view:", VIEW)
print("secret scope:", SECRET_SCOPE)
print("credentials:", CREDENTIALS_PATH)

results: list[tuple[str, bool, str]] = []
timings: list[tuple[str, float | None, int, bool]] = []


def check(name: str, ok: bool, detail: str = "") -> bool:
    results.append((name, ok, detail))
    mark = "PASS" if ok else "FAIL"
    suffix = f"  — {detail}" if detail else ""
    print(f"{mark}  {name}{suffix}")
    return ok


def skip(name: str, reason: str) -> None:
    print(f"skip  {name}  — {reason}")


## 3. CDF client


In [ ]:
from cognite.client import CogniteClient
from cognite.client.config import ClientConfig
from cognite.client.credentials import OAuthClientCredentials

with Path(CREDENTIALS_PATH).open("rb") as handle:
    data = tomllib.load(handle)
cognite = data["cognite"]
client_id = cognite.get("client_id")
client_secret = cognite.get("client_secret")
if not client_id or not client_secret:
    for section in data.values():
        if not isinstance(section, dict):
            continue
        client_id = client_id or section.get("client_id")
        client_secret = client_secret or section.get("client_secret")
tenant_id = cognite.get("tenant_id") or cognite.get("idp_tenant_id")
if not client_id or not client_secret or not tenant_id:
    raise ValueError("TOML is missing client_id, client_secret, or tenant_id")

base_url = cognite.get("base_url") or f"https://{cognite['cdf_cluster']}.cognitedata.com"
client = CogniteClient(
    ClientConfig(
        client_name="pushdown-latency",
        project=cognite["project"],
        credentials=OAuthClientCredentials(
            token_url=f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
            client_id=client_id,
            client_secret=client_secret,
            scopes=[f"https://{cognite['cdf_cluster']}.cognitedata.com/.default"],
        ),
        base_url=base_url.rstrip("/"),
    )
)
token = client.iam.token.inspect()
print("project:", client.config.project, "| base_url:", client.config.base_url)
check("CDF client authenticates", token is not None, client.config.base_url)


## 4. Load the data model

`rewrite_query` needs the view metadata. Registration is left as it is.


In [ ]:
from cognite.client.data_classes.data_modeling.ids import DataModelId
from cognite.databricks import generate_udtf_notebook
from databricks.sdk import WorkspaceClient

generator = generate_udtf_notebook(
    DataModelId(space=DATA_MODEL_SPACE, external_id=DATA_MODEL_EXTERNAL_ID, version=DATA_MODEL_VERSION),
    client,
    workspace_client=WorkspaceClient(),
    catalog=CATALOG,
    schema=SCHEMA,
    output_dir=OUTPUT_DIR,
)
print("generator ready for", VIEW)


## 5. One query per pushdown filter

Run the warm-up cell first. It starts Spark and loads the UDTF worker. That time is printed and then discarded.


In [ ]:
import re
import time


def sql_string_list(values: list[str]) -> str:
    return ", ".join("'" + value.replace("'", "''") + "'" for value in values)


def selected_columns() -> str:
    columns = ["space", "external_id"]
    for column in (STRING_COLUMN, NULLABLE_COLUMN, NUMERIC_COLUMN):
        if column and column not in columns:
            columns.append(column)
    return ", ".join(columns)


def returned(_rows: list) -> bool:
    return True


def pushed_bindings(statement: str) -> str:
    bindings = []
    for name, value in re.findall(r"(\w+)\s*=>\s*([^,\n]+)", statement):
        if name in {"client_id", "client_secret", "tenant_id", "cdf_cluster", "project", "base_url"}:
            continue
        if value.strip() == "NULL":
            continue
        bindings.append(f"{name}={value.strip()}")
    return ", ".join(bindings)


def measure(name: str, analyst_sql: str, expect=returned) -> None:
    print(f"\n=== {name} ===")
    print(analyst_sql.strip())
    try:
        rewritten = generator.rewrite_query(analyst_sql, secret_scope=SECRET_SCOPE)
        if rewritten is None:
            timings.append((name, None, 0, False))
            check(name, False, "not pushed down")
            return
        print("pushed:", pushed_bindings(rewritten))
        frame = spark.sql(rewritten)
        started = time.perf_counter()
        rows = frame.collect()
        elapsed = time.perf_counter() - started
        if rows:
            result = spark.createDataFrame(rows, schema=frame.schema)
            result.show(20, truncate=False)
            display(result)
        else:
            print("(no rows)")
        print(f"{elapsed:.2f}s  {len(rows)} row(s)")
        ok = bool(expect(rows))
        timings.append((name, elapsed, len(rows), ok))
        check(name, ok, f"{elapsed:.2f}s  {len(rows)} row(s)")
    except Exception as exc:
        timings.append((name, None, 0, False))
        check(name, False, f"{type(exc).__name__}: {exc}"[:300])


### 5.0 Warm-up


In [ ]:
print("Warm-up. This time is not recorded.")
space_filter = f" WHERE space = '{INSTANCE_SPACES[0]}'" if INSTANCE_SPACES else ""
started = time.perf_counter()
spark.sql("SELECT 1 AS spark_ready").collect()
warmup_sql = generator.rewrite_query(
    f"SELECT external_id FROM {VIEW}{space_filter} LIMIT 1",
    secret_scope=SECRET_SCOPE,
)
if warmup_sql is None:
    raise RuntimeError("warm-up query was not pushed down")
spark.sql(warmup_sql).collect()
print(f"warm-up {time.perf_counter() - started:.2f}s")


### 5.1 Instance space equals


In [ ]:
if not INSTANCE_SPACES:
    skip("Instance space equals", "set INSTANCE_SPACES")
else:
    measure(
        "Instance space equals",
        f"SELECT count(*) AS n FROM {VIEW} WHERE space = '{INSTANCE_SPACES[0]}'",
    )


### 5.2 Instance space IN


In [ ]:
if len(INSTANCE_SPACES) < 2:
    skip("Instance space IN", "set at least two INSTANCE_SPACES")
else:
    measure(
        "Instance space IN",
        f"SELECT count(*) AS n FROM {VIEW} WHERE space IN ({sql_string_list(INSTANCE_SPACES[:2])})",
    )


### 5.3 View space


In [ ]:
measure(
    "View space",
    f"SELECT count(*) AS n FROM {VIEW} WHERE space = '{DATA_MODEL_SPACE}'",
)


### 5.4 external_id equals


In [ ]:
if not EXTERNAL_IDS:
    skip("external_id equals", "set EXTERNAL_IDS")
else:
    measure(
        "external_id equals",
        f"SELECT {selected_columns()} FROM {VIEW} WHERE external_id = '{EXTERNAL_IDS[0]}'",
    )


### 5.5 external_id IN


In [ ]:
if len(EXTERNAL_IDS) < 2:
    skip("external_id IN", "set at least two EXTERNAL_IDS")
else:
    measure(
        "external_id IN",
        f"SELECT {selected_columns()} FROM {VIEW} WHERE external_id IN ({sql_string_list(EXTERNAL_IDS[:2])})",
    )


### 5.6 Property equals


In [ ]:
if not STRING_COLUMN or not STRING_VALUES:
    skip("Property equals", "set STRING_COLUMN and STRING_VALUES")
else:
    measure(
        "Property equals",
        f"SELECT count(*) AS n FROM {VIEW} WHERE {STRING_COLUMN} = '{STRING_VALUES[0]}'",
    )


### 5.7 Property IN


In [ ]:
if not STRING_COLUMN or len(STRING_VALUES) < 2:
    skip("Property IN", "set STRING_COLUMN and at least two STRING_VALUES")
else:
    measure(
        "Property IN",
        f"SELECT count(*) AS n FROM {VIEW} WHERE {STRING_COLUMN} IN ({sql_string_list(STRING_VALUES[:2])})",
    )


### 5.8 IS NOT NULL (exists)


In [ ]:
if not NULLABLE_COLUMN:
    skip("IS NOT NULL (exists)", "set NULLABLE_COLUMN")
else:
    measure(
        "IS NOT NULL (exists)",
        f"SELECT count(*) AS n FROM {VIEW} WHERE {NULLABLE_COLUMN} IS NOT NULL",
    )


### 5.9 IS NULL (not exists)


In [ ]:
if not NULLABLE_COLUMN:
    skip("IS NULL (not exists)", "set NULLABLE_COLUMN")
else:
    measure(
        "IS NULL (not exists)",
        f"SELECT count(*) AS n FROM {VIEW} WHERE {NULLABLE_COLUMN} IS NULL",
    )


### 5.10 Range >


In [ ]:
if not NUMERIC_COLUMN or NUMERIC_HIGH is None:
    skip("Range >", "set NUMERIC_COLUMN and NUMERIC_HIGH")
else:
    measure(
        "Range >",
        f"SELECT count(*) AS n FROM {VIEW} WHERE {NUMERIC_COLUMN} > {NUMERIC_HIGH}",
    )


### 5.11 Range >=


In [ ]:
if not NUMERIC_COLUMN or NUMERIC_HIGH is None:
    skip("Range >=", "set NUMERIC_COLUMN and NUMERIC_HIGH")
else:
    measure(
        "Range >=",
        f"SELECT count(*) AS n FROM {VIEW} WHERE {NUMERIC_COLUMN} >= {NUMERIC_HIGH}",
    )


### 5.12 Range <


In [ ]:
if not NUMERIC_COLUMN or NUMERIC_LOW is None:
    skip("Range <", "set NUMERIC_COLUMN and NUMERIC_LOW")
else:
    measure(
        "Range <",
        f"SELECT count(*) AS n FROM {VIEW} WHERE {NUMERIC_COLUMN} < {NUMERIC_LOW}",
    )


### 5.13 Range between


In [ ]:
if not NUMERIC_COLUMN or NUMERIC_LOW is None or NUMERIC_HIGH is None:
    skip("Range between", "set NUMERIC_COLUMN, NUMERIC_LOW, and NUMERIC_HIGH")
else:
    measure(
        "Range between",
        f"SELECT count(*) AS n FROM {VIEW} WHERE {NUMERIC_COLUMN} >= {NUMERIC_LOW} AND {NUMERIC_COLUMN} <= {NUMERIC_HIGH}",
    )


### 5.14 LIMIT


In [ ]:
space_filter = f" WHERE space = '{INSTANCE_SPACES[0]}'" if INSTANCE_SPACES else ""
measure(
    "LIMIT",
    f"SELECT {selected_columns()} FROM {VIEW}{space_filter} LIMIT 2",
)


### 5.15 COUNT(*)


In [ ]:
measure("COUNT(*)", f"SELECT count(*) AS n FROM {VIEW}")


### 5.16 Numeric MIN


In [ ]:
if not NUMERIC_COLUMN:
    skip("Numeric MIN", "set NUMERIC_COLUMN")
else:
    space_filter = f" WHERE space = '{INSTANCE_SPACES[0]}'" if INSTANCE_SPACES else ""
    measure(
        "Numeric MIN",
        f"SELECT min({NUMERIC_COLUMN}) AS low FROM {VIEW}{space_filter}",
    )


### 5.17 Numeric MAX


In [ ]:
if not NUMERIC_COLUMN:
    skip("Numeric MAX", "set NUMERIC_COLUMN")
else:
    space_filter = f" WHERE space = '{INSTANCE_SPACES[0]}'" if INSTANCE_SPACES else ""
    measure(
        "Numeric MAX",
        f"SELECT max({NUMERIC_COLUMN}) AS high FROM {VIEW}{space_filter}",
    )


### 5.18 GROUP BY


In [ ]:
if not STRING_COLUMN:
    skip("GROUP BY", "set STRING_COLUMN")
else:
    measure(
        "GROUP BY",
        f"SELECT {STRING_COLUMN}, count(*) AS n FROM {VIEW} GROUP BY {STRING_COLUMN}",
    )


### 5.19 MIN on text (not pushed)


In [ ]:
if not STRING_COLUMN:
    skip("MIN on text", "set STRING_COLUMN")
else:
    text_min = f"SELECT min({STRING_COLUMN}) AS low FROM {VIEW}"
    print(text_min)
    rewritten = generator.rewrite_query(text_min, secret_scope=SECRET_SCOPE)
    print("rewritten:", rewritten)
    check("MIN on text stays in Spark", rewritten is None)


## 6. Latency summary


In [ ]:
if not timings:
    print("No timed queries. Fill in the filter settings and rerun section 5.")
else:
    summary = spark.createDataFrame(
        [
            (name, None if seconds is None else round(seconds, 2), row_count, "PASS" if ok else "FAIL")
            for name, seconds, row_count, ok in timings
        ],
        "filter string, seconds double, rows long, result string",
    )
    summary.show(30, truncate=False)
    display(summary)
failed = [name for name, ok, _detail in results if not ok]
print(f"{len(results) - len(failed)} passed, {len(failed)} failed")
if failed:
    print("failed:", ", ".join(failed))
